In [5]:
import math
import numpy as np
import pandas as pd
from pathlib import Path

PROFILE_SECONDS = 15.0
system = "V100"
# system = "A100"
# system = "H100"

DATA_DIR = Path(f'/home/ac.zzheng/power/GPGPU/coSched/data/{system}')
power_cap = 2800
gpu_counts = [1, 2, 3, 4]

spec_dir = DATA_DIR / 'spec_power_motif'
ml_dir = DATA_DIR / 'ml_power_motif'
cuda_dir = DATA_DIR / 'cuda_power_motif'
spec_benchmarks = [d.name for d in spec_dir.iterdir() if d.is_dir()]
ml_benchmarks = [d.name for d in ml_dir.iterdir() if d.is_dir()]
cuda_benchmarks = [d.name for d in cuda_dir.iterdir() if d.is_dir()]
all_configs = [(spec_dir, spec_benchmarks), (ml_dir, ml_benchmarks), (cuda_dir, cuda_benchmarks)]


def extract_metrics_from_csv(csv_path, gpu_count):
    df = pd.read_csv(csv_path)
    if len(df) < 2:
        return None
    dram_cols = [f'GPU{i}_DRAM_Active' for i in range(gpu_count) if f'GPU{i}_DRAM_Active' in df.columns]

    if dram_cols:
        dram_df = df[dram_cols].fillna(0.0)
        has_any_nonzero_dram = dram_df.ne(0.0).any(axis=1).any()
        if has_any_nonzero_dram:
            df = df[dram_df.ne(0.0).any(axis=1)]

    if len(df) == 0:
        return None

    t0 = df['Time (s)'].min()
    df = df[df['Time (s)'] <= t0 + PROFILE_SECONDS]
    if len(df) == 0:
        return None
    active = [i for i in range(gpu_count) if f'GPU{i}_Power (W)' in df.columns]
    if not active:
        return None

    p_avg, dr_avg, sm_avg = {}, {}, {}
    fp16_avg, fp32_avg, fp64_avg = {}, {}, {}
    for gid in active:
        p_avg[gid] = df[f'GPU{gid}_Power (W)'].mean()
        sm_avg[gid] = df[f'GPU{gid}_SM_Clock (MHz)'].mean() if f'GPU{gid}_SM_Clock (MHz)' in df.columns else 0
        dr_avg[gid] = df[f'GPU{gid}_DRAM_Active'].mean() if f'GPU{gid}_DRAM_Active' in df.columns else 0
        fp16_avg[gid] = df[f'GPU{gid}_FP16_Active'].mean() if f'GPU{gid}_FP16_Active' in df.columns else 0
        fp32_avg[gid] = df[f'GPU{gid}_FP32_Active'].mean() if f'GPU{gid}_FP32_Active' in df.columns else 0
        fp64_avg[gid] = df[f'GPU{gid}_FP64_Active'].mean() if f'GPU{gid}_FP64_Active' in df.columns else 0

    def safe(v):
        return 0.0 if (v is None or (isinstance(v, float) and math.isnan(v))) else v

    dr_list = [safe(dr_avg[g]) for g in active]
    sm_list = [safe(sm_avg[g]) for g in active]
    fp_list = [safe(fp16_avg[g]) + safe(fp32_avg[g]) + safe(fp64_avg[g]) for g in active]

    return {
        'avg_power': safe(np.mean([safe(p_avg[g]) for g in active])),
        'dram_sum': sum(dr_list),
        'sm_sum': sum(sm_list),
        'fp_sum': sum(fp_list),
    }


def get_runtime_from_csv(csv_path):
    """Get runtime as max Time (s) from gpu_metrics CSV."""
    df = pd.read_csv(csv_path)
    if 'Time (s)' not in df.columns or len(df) == 0:
        return float('nan')
    return df['Time (s)'].max()


txt_lines = []
for base_dir, benchmarks in all_configs:
    suite = 'SPEC' if 'spec' in str(base_dir) else 'ML'
    for app in sorted(benchmarks):
        app_lines = []
        for gc in gpu_counts:
            csv_file = base_dir / app / f"{power_cap}_{gc}_gpu_metrics.csv"
            if not csv_file.exists():
                continue
            metrics = extract_metrics_from_csv(csv_file, gc)
            if metrics is None:
                continue
            perf = get_runtime_from_csv(csv_file)
            app_lines.append((gc, perf, metrics))

        if app_lines:
            # Sort by performance (runtime ascending = best first)
            app_lines.sort(key=lambda x: x[1])
            txt_lines.append(f"\n===== {suite}/{app} =====")
            txt_lines.append(f"cap={power_cap}")
            txt_lines.append("gpu_count\tperformance\tavg_power\tdram_sum\tsm_sum\tfp_sum")
            for gc, perf, m in app_lines:
                txt_lines.append(
                    f"{gc}\t{perf:.2f}\t{m['avg_power']:.2f}\t{m['dram_sum']:.3f}\t{m['sm_sum']:.2f}\t{m['fp_sum']:.2f}"
                )

txt_output = '\n'.join(txt_lines)
txt_path = DATA_DIR / 'perf_metrics.txt'
txt_path.write_text(txt_output)
print(txt_output)
print(f"\n[written] {txt_path}")


===== SPEC/cloverleaf =====
cap=2800
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum
4	37.17	174.46	2.929	6120.00	0.00
3	49.61	176.94	2.156	4590.00	0.00
2	67.56	187.62	1.577	3060.00	0.00
1	128.41	204.53	0.826	1530.00	0.00

===== SPEC/hpgmg =====
cap=2800
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum
1	19.26	172.64	0.340	1512.86	0.00
3	34.93	137.80	0.662	4590.00	0.00
2	44.45	171.20	0.731	3060.00	0.00
4	80.73	131.00	1.081	6072.00	0.00

===== SPEC/lbm =====
cap=2800
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum
4	21.73	236.28	2.458	6105.41	0.00
3	28.25	235.47	1.805	4561.76	0.00
2	40.59	247.47	1.302	3060.00	0.00
1	86.45	250.76	0.661	1530.00	0.00

===== SPEC/minisweep =====
cap=2800
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum
4	65.30	87.64	0.306	6120.00	0.00
3	88.57	88.96	0.247	4590.00	0.00
2	121.61	92.21	0.171	3060.00	0.00

===== SPEC/miniweather =====
cap=2800
gpu_count	performance	avg_power	dram_sum	sm_sum	fp_sum
4	111.37	138.13	0.615	6120.00	0.00
3	1